In [ ]:
!pip install -q faster-whisper fastapi uvicorn python-multipart pyngrok torch torchaudio

In [ ]:
# DeepFilterNet3 (pip install deepfilternet) é leve, real-time e roda até em
# CPU; reduz ruído forte sem os artefatos do resemble-enhance e sem as
# dependências travadas de 2023 (torch==2.1.1, numpy==1.26.2, deepspeed etc.).
# O modelo baixa automaticamente (~8MB) do GitHub no primeiro /enhance.
#
# Se essa célula falhar, o resto do notebook (pyngrok, uvicorn, /transcribe)
# continua funcionando — só o endpoint /enhance ficaria indisponível até
# você resolver o erro daqui.
!pip install -q deepfilternet

In [ ]:
%%writefile app.py
import os
import tempfile
import time
from contextlib import suppress
from typing import Optional

import soundfile as sf
import torch
from fastapi import BackgroundTasks, Depends, FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from faster_whisper import WhisperModel

TRANSCRIBE_MODEL_NAME = "large-v3-turbo"
ENHANCE_MODEL_NAME = "DeepFilterNet3"
MAX_UPLOAD_MB = 500

# DeepFilterNet3 é carregado sob demanda (lazy) na primeira chamada a
# /enhance. Cache global pra não recarregar o modelo a cada request:
_DF = None  # (model, df_state)

API_TOKEN = os.environ["API_TOKEN"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"


def _remove_file(path: str) -> None:
    with suppress(OSError):
        os.remove(path)

app = FastAPI(
    title="Speech API",
    version="1.0.0"
)

# ==========================
# CORS
# ==========================

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================
# AUTH
# ==========================

bearer_scheme = HTTPBearer()


def require_api_token(
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
) -> None:
    if credentials.credentials != API_TOKEN:
        raise HTTPException(status_code=401, detail="Invalid or missing API token")


print(f"Carregando modelo de transcrição {TRANSCRIBE_MODEL_NAME} ({DEVICE}/{COMPUTE_TYPE})...")

transcribe_model = WhisperModel(
    TRANSCRIBE_MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

print("Modelo de transcrição carregado!")

# O modelo de enhance é carregado sob demanda dentro do endpoint /enhance
# (não aqui no startup): assim, /transcribe e /enhance ficam independentes
# entre si — se o DeepFilterNet3 falhar ao carregar/baixar pesos, isso
# não derruba a API de transcrição, e vice-versa.

# ==========================
# ROOT
# ==========================

@app.get("/")
def root():
    return {
        "name": "Speech API",
        "status": "online",
        "version": "1.0.0"
    }

# ==========================
# HEALTH
# ==========================

@app.get("/api/health")
def health():
    return {
        "status": "ok",
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "version": "1.0.0",
        "endpoints": {
            "transcribe": {"model": TRANSCRIBE_MODEL_NAME, "loaded": True},
            "enhance": {"model": ENHANCE_MODEL_NAME, "loaded": "lazy"}
        }
    }

# ==========================
# TRANSCRIBE
# ==========================

@app.post("/transcribe", dependencies=[Depends(require_api_token)])
def transcribe(
    file: UploadFile = File(...),
    language: str = Form("pt"),
    vad: bool = Form(True),
    word_timestamps: bool = Form(True)
):
    start = time.time()

    suffix = os.path.splitext(file.filename)[1]
    content = file.file.read()

    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File too large (max {MAX_UPLOAD_MB}MB)",
        )

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        path = tmp.name

    try:
        segments, info = transcribe_model.transcribe(
            path,
            language=None if language in ("", "auto") else language,
            vad_filter=vad,
            word_timestamps=word_timestamps
        )

        result_segments = []
        full_text = []

        words_count = 0

        duration = 0

        for seg in segments:

            duration = seg.end

            full_text.append(seg.text)

            words = []

            if seg.words:

                for w in seg.words:

                    words_count += 1

                    words.append({
                        "word": w.word,
                        "start": w.start,
                        "end": w.end
                    })

            result_segments.append({

                "start": seg.start,
                "end": seg.end,
                "text": seg.text,
                "words": words

            })
    except Exception as exc:
        raise HTTPException(
            status_code=422, detail=f"Transcription failed: {exc}"
        ) from exc
    finally:
        with suppress(OSError):
            os.remove(path)

    return {

        "success": True,

        "language": info.language,

        "duration": duration,

        "processing_time": round(time.time() - start, 2),

        "model": TRANSCRIBE_MODEL_NAME,

        "segments": result_segments,

        "segments_count": len(result_segments),

        "words_count": words_count,

        "text": "".join(full_text)

    }

# ==========================
# ENHANCE
# ==========================

@app.post("/enhance", dependencies=[Depends(require_api_token)])
def enhance(
    file: UploadFile = File(...),
    denoise_only: bool = Form(False),
    atten_lim_db: Optional[float] = Form(None),
):
    import torchaudio

    # deepfilternet 0.5.6 (PyPI, 2023) importa torchaudio.backend.common e
    # usa torchaudio.info, que sumiram no torchaudio >= 2.4 (Colab instala o
    # mais novo). Shim de compatibilidade antes de importar o df:
    import sys as _sys
    import types as _types
    if not hasattr(torchaudio, "info"):
        import soundfile as _sf

        class _AudioMetaData:
            def __init__(self, path):
                self._i = _sf.info(path)

            @property
            def sample_rate(self):
                return self._i.samplerate

            @property
            def num_frames(self):
                return int(self._i.frames)

            @property
            def num_channels(self):
                return self._i.channels

        torchaudio.info = lambda path, **kw: _AudioMetaData(path)
        _backend = _types.ModuleType("torchaudio.backend")
        _common = _types.ModuleType("torchaudio.backend.common")
        _common.AudioMetaData = _AudioMetaData
        _backend.common = _common
        _sys.modules["torchaudio.backend"] = _backend
        _sys.modules["torchaudio.backend.common"] = _common

    from df.enhance import enhance as df_enhance
    from df.enhance import init_df
    from df.io import resample

    global _DF
    if _DF is None:
        # PyPI 0.5.6 retorna (model, df_state, suffix)
        _DF = init_df(ENHANCE_MODEL_NAME, log_level="WARNING")

    df_model, df_state = _DF[0], _DF[1]

    start = time.time()

    suffix = os.path.splitext(file.filename)[1]
    content = file.file.read()

    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File too large (max {MAX_UPLOAD_MB}MB)",
        )

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        input_path = tmp.name

    output_path = f"{input_path}_enhanced.flac"

    try:
        # DeepFilterNet opera em 48kHz (df_state.sr()). Carrega e resamplea
        # pra taxa do modelo; depois volta pra taxa original do input.
        audio, orig_sr = torchaudio.load(input_path)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        df_sr = df_state.sr()
        if orig_sr != df_sr:
            audio = resample(audio, orig_sr, df_sr)

        # denoise_only → default do DeepFilterNet (limite de atenuação ~10dB).
        if denoise_only:
            atten_lim_db = 10.0 if atten_lim_db is None else atten_lim_db

        enhanced_wav = df_enhance(
            df_model,
            df_state,
            audio,
            atten_lim_db=atten_lim_db,
        )

        if orig_sr != df_sr:
            enhanced_wav = resample(enhanced_wav, df_sr, orig_sr)

        sf.write(
            output_path,
            enhanced_wav.squeeze(0).cpu().numpy(),
            orig_sr,
            format="FLAC",
        )
    except Exception as exc:
        import traceback as _tb
        print("\n===== /enhance TRACEBACK =====", flush=True)
        _tb.print_exc()
        with suppress(OSError):
            os.remove(output_path)
        raise HTTPException(
            status_code=422,
            detail={
                "error": "Enhance failed",
                "message": str(exc),
                "type": type(exc).__name__,
                "traceback": _tb.format_exc(),
            },
        ) from exc
    finally:
        with suppress(OSError):
            os.remove(input_path)

    cleanup = BackgroundTasks()
    cleanup.add_task(_remove_file, output_path)

    return FileResponse(
        output_path,
        media_type="audio/flac",
        filename="enhanced.flac",
        background=cleanup,
        headers={
            "X-Model": ENHANCE_MODEL_NAME,
            "X-Processing-Time": str(round(time.time() - start, 2)),
        },
    )


In [ ]:
import os

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# Crie um secret "API_TOKEN" no Colab (ícone de chave na barra lateral) com
# um valor aleatório seu — ele protege os endpoints /transcribe e /enhance
# de uso por qualquer pessoa que descubra a URL pública do ngrok.
os.environ['API_TOKEN'] = userdata.get('API_TOKEN')

public_url = ngrok.connect(8000)

print(public_url)
print('Header em ambos os endpoints: Authorization: Bearer <seu API_TOKEN>')
print('POST /transcribe -> transcrição (faster-whisper)')
print('POST /enhance    -> speech enhancement (DeepFilterNet3), independente do /transcribe')

In [ ]:
import subprocess
import time

log_file = open('uvicorn.log', 'w')
uvicorn_process = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(3)
print(f'Uvicorn iniciado (PID {uvicorn_process.pid}). Logs em uvicorn.log')
print(f'Ainda rodando: {uvicorn_process.poll() is None}')
# Pra ver os logs a qualquer momento: !tail -n 50 uvicorn.log
# Pra derrubar o servidor: uvicorn_process.terminate()